# Notebook 6: Generate Running Log Completions

**Goal**: Explore inference-time behavior. Compare decoding strategies.
Visualize attention patterns. Benchmark the trained model.

Run this notebook after completing Notebook 5 (or after running `train.py`).

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import math
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

from 02_tokenization.bpe_tokenizer import BPETokenizer
from 05_transformer.gpt_model import RunningGPT
from 07_inference.generate import generate, greedy_generate
from 07_inference.beam_search import beam_search

torch.manual_seed(0)
print('Imports OK')
print('Note: This notebook assumes you have a trained model checkpoint.')
print('If not, run Notebook 5 first to train a quick demo model.')

## 1. Quick in-notebook training (if no checkpoint available)

In [ ]:
# Train a quick model right in this notebook for demo purposes
from 01_data.generate_synthetic import SyntheticRunGenerator
from 01_data.data_loader import RunningDataset, build_dataloaders
from 06_training.optimizer import build_optimizer, CosineWarmupScheduler
from 06_training.loss import cross_entropy_loss

gen = SyntheticRunGenerator(seed=0)
corpus = gen.generate_run_notes(2000)

tok = BPETokenizer(vocab_size=800)
tok.train(corpus, verbose=False)

all_ids = []
for doc in corpus:
    all_ids.extend(tok.encode(doc, add_special=True))

dataset = RunningDataset(all_ids, context_len=64)
train_loader, _ = build_dataloaders(dataset, batch_size=16)

model = RunningGPT(vocab_size=len(tok), d_model=64, num_heads=2, num_layers=2,
                   d_ff=128, max_seq_len=64, dropout=0.1)

optimizer = build_optimizer(model, lr=3e-4)
scheduler = CosineWarmupScheduler(optimizer, warmup_steps=50, max_steps=800)

data_iter = iter(train_loader)
model.train()
optimizer.zero_grad()

for step in range(800):
    try:
        x, y = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        x, y = next(data_iter)
    logits, _ = model(x)
    loss = cross_entropy_loss(logits, y)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step(); scheduler.step(); optimizer.zero_grad()
    if step % 200 == 0:
        print(f'step {step}: loss={loss.item():.4f}')

model.eval()
print('Model trained and ready!')

## 2. Compare decoding strategies side by side

In [ ]:
prompt = 'easy run today'
ids = tok.encode(prompt)
x = torch.tensor(ids).unsqueeze(0)

results = {}

# Greedy
out = greedy_generate(model, x.clone(), max_new_tokens=25)
results['Greedy'] = tok.decode(out[0, len(ids):].tolist())

# Top-K
for k in [5, 20, 50]:
    out = generate(model, x.clone(), max_new_tokens=25, temperature=1.0, top_k=k)
    results[f'Top-K (k={k})'] = tok.decode(out[0, len(ids):].tolist())

# Nucleus
for p in [0.7, 0.9, 0.95]:
    out = generate(model, x.clone(), max_new_tokens=25, temperature=1.0, top_p=p)
    results[f'Nucleus (p={p})'] = tok.decode(out[0, len(ids):].tolist())

# Temperature
for temp in [0.5, 1.0, 1.5]:
    out = generate(model, x.clone(), max_new_tokens=25, temperature=temp, top_k=40)
    results[f'Temp={temp}'] = tok.decode(out[0, len(ids):].tolist())

print(f'Prompt: "{prompt}"\n')
for strategy, completion in results.items():
    print(f'{strategy:<20}: {completion}')

## 3. Beam search vs sampling

In [ ]:
prompt = 'tempo run 8 miles at'
ids = tok.encode(prompt)
x = torch.tensor(ids).unsqueeze(0)

# Beam search
beam_ids = beam_search(model, x.clone(), beam_size=4, max_new_tokens=20, eos_id=tok.eos_id)
beam_out = tok.decode(beam_ids[len(ids):])

# Nucleus sampling (run 3 times to show diversity)
print(f'Prompt: "{prompt}"\n')
print(f'Beam search:   {beam_out}')
for i in range(3):
    out = generate(model, x.clone(), max_new_tokens=20, temperature=0.9, top_p=0.9)
    sample_out = tok.decode(out[0, len(ids):].tolist())
    print(f'Nucleus [{i+1}]:    {sample_out}')

## 4. Visualize attention weights

In [ ]:
prompt = 'marathon pace was strong today'
ids = tok.encode(prompt)
x = torch.tensor(ids).unsqueeze(0)
tokens = [tok.vocab.get(i, '?').replace('Ġ', '') for i in ids]

with torch.no_grad():
    _, all_weights = model(x, return_all_weights=True)

# Plot attention for first layer, all heads
fig, axes = plt.subplots(1, model.num_layers * 2, figsize=(5 * model.num_layers * 2, 4))

col = 0
for layer_idx, layer_weights in enumerate(all_weights):
    num_heads = layer_weights.shape[1]
    for h in range(num_heads):
        ax = axes[col]
        data = layer_weights[0, h, :len(tokens), :len(tokens)].detach().numpy()
        im = ax.imshow(data, cmap='Blues', aspect='auto', vmin=0)
        ax.set_xticks(range(len(tokens)))
        ax.set_yticks(range(len(tokens)))
        ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
        ax.set_yticklabels(tokens, fontsize=8)
        ax.set_title(f'L{layer_idx+1} H{h+1}', fontsize=9)
        col += 1

plt.suptitle(f'Attention weights: "{prompt}"', y=1.02)
plt.tight_layout()
plt.show()

## 5. Qualitative generation test

Generate 10 completions for different running contexts.

In [ ]:
test_prompts = [
    'easy recovery run',
    'marathon race day',
    'interval workout on the track',
    'trail run with elevation',
    'heart rate was',
    'pace per mile',
    'felt strong during',
    'weather was',
    'personal record',
    'training week total',
]

print('Generated running log completions:\n')
print('=' * 65)
for prompt in test_prompts:
    ids = tok.encode(prompt)
    x = torch.tensor(ids).unsqueeze(0)
    out = generate(model, x, max_new_tokens=20, temperature=0.85, top_p=0.9)
    completion = tok.decode(out[0, len(ids):].tolist())
    print(f'{prompt} {completion}')
print('=' * 65)

## Exercise

1. Count how many of the 10 completions above are grammatically reasonable.
   What % success rate do you get? How does this improve after more training?
2. Try prompts the model has never seen (make up a very specific route name).
   How does it handle out-of-distribution inputs?
3. Generate 50 completions for 'my marathon time was'. Build a histogram of the
   generated finishing times. Do they follow a realistic distribution?
4. Compute the perplexity on your own real running logs (if you have them).
   Is it lower or higher than on the synthetic test set? What does that tell you?